# Somo la 18: Kuweka Usalama wa Wakala wa AI kwa Kupokea kwa Usimbaji wa Kielektroniki

## Daftari la Vitendo

Daftari hili linafikisha mazoezi manne:

1. **Sainisha risiti yako ya kwanza** kwa wito wa chombo cha wakala na uithibitishe.
2. **Fanya uharibifu kwenye risiti** na utaona uthibitisho kushindwa.
3. **Jenga mnyororo wa risiti tatu** na thibitisha uadilifu wa mnyororo.
4. **Funga wito wa chombo cha Microsoft Agent Framework** ili kila hatua itoe risiti.

Vifaa vyote vya usimbaji huletwa kutoka maktaba zinazotunzwa vizuri (`pynacl` kwa Ed25519, `jcs` kwa JSON la RFC 8785 la kawaida, `hashlib` kutoka maktaba ya kawaida ya Python kwa SHA-256). Mantiki ya risiti mwenyewe ni Python rahisi ambayo unaweza kusoma na kubadilisha.

Endesha seli kwa mpangilio. Kila sehemu ni fupi na lina uhuru wake.


## Usanidi

Sakinisha utegemezi wawili. Zote zina leseni zinazoruhusu (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Vifaa Msaidizi

Hawa wasaidizi wawili hushughulikia usimbaji wa base64url (bila kujaza) na upigo wa SHA-256 wa vitu vya aina yoyote. Huweka sehemu nyingine ya daftari la kumbukumbu ikilenga moja kwa moja mantiki ya risiti.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sehemu 1: Saini risiti yako ya kwanza

Fikiria wakala wetu wa **Contoso Travel** alichunguza ndege kutoka Sydney hadi Los Angeles kwa mwanunuzi. Tunataka kurekodi simu hii ya zana kama risiti iliyosainiwa ili mkaguzi wa baadaye aweze kuithibitisha bila kutuamini.

### Hatua 1.1: Tengeneza ufunguo wa kusaini

Katika utengenezaji, ufunguo wa kusaini wa wakala ungeishi katika kipengee cha usalama cha vifaa (HSM), Azure Key Vault, au duka linalofanana lililolindwa. Kwa somo hili tunatengeneza ufunguo mpya katika kumbukumbu.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Hatua 1.2: Tengeneza mzigo wa risiti

Mzigo una kila kitu tunachotaka risiti ithibitishe: nani alitenda, chombo gani, kwa hoja gani, kilichorejea nini, chini ya sera gani, na lini. Tunafanya hash ya hoja na matokeo badala ya kuziweka moja kwa moja ili risiti isitokeze maudhui nyeti.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Hatua 1.3: Saini na kukusanya risiti

Hatua tatu:

1. Fanya payload kuwa rasmi kwa kutumia JCS hivyo utekelezaji wawili wanaotoa risiti yenye mantiki sawa watengeneze baiti zinazolingana kabisa.
2. Fanya hash ya baiti rasmi kwa kutumia SHA-256.
3. Saini hash kwa kutumia ufunguo wa siri wa Ed25519.

Saini kisha inafungwa kwenye payload asilia ili kuzalisha risiti ya mwisho.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Hatua 1.4: Hakiki risiti

Ukaguzi unarekebisha mchakato. Tunatoa saini, kuhesabu upya siginecha ya kawaida, na kukagua saini dhidi ya ufunguo wa umma kwenye risiti.

Mhakiki anayefanya ukaguzi huu hahitaji chochote kutoka kwetu isipokuwa risiti yenyewe. Hakuna huduma ya kuitisha, hakuna saraka ya funguo ya kuulizia, hakuna kuaminika kunahitajika.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Unapaswa kuona `Receipt is valid: True`. Wakala ametoa rekodi yake ya ukaguzi iliyosainiwa kifalsafi kwa mara ya kwanza.


## Sehemu 2: Badilisha risiti

Kitu kikuu cha risiti ni kwamba zinathibitisha wazi zinapobadilishwa. Hebu tuonyeshe.

Tutabadilisha herufi moja tu ya risiti na tazama uthibitishaji ukishindwa.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Nini kilitokea tu?

Tulipobadilisha `policy_id`, baiti za kawaida zilibadilika. Hashi ya SHA-256 ya baiti hizo ilibadilika. Saini (ambayo ilikuwa juu ya hashi ya awali) haikilingani tena na hashi mpya. Uthibitisho unarudisha `False` kwa usahihi.

Hakuna njia ya kubadilisha sehemu yoyote ya risiti na bado kuthibitisha, isipokuwa mdukuji ana ufunguo wa kibinafsi. Kadri ufunguo wa kibinafsi ulivyo katika ghala la funguo na ufunguo wa umma umetangazwa, kujaribu kubadilisha ni vigumu kuficha.

Jaribu wewe mwenyewe: badilisha `tool_name` au `agent_id` au `timestamp` kwenye seli hapo juu na uendelee kurusha tena. Kila mabadiliko huleta risiti isiyo halali.


## Sehemu ya 3: Kuunganisha risiti pamoja

Risiti moja hutunza hatua moja. Wakala wengi huchukua hatua nyingi. Ili kufanya mfululizo mzima kuwa mguso wa uhuru wa usalama, tunahusisha kila risiti na ile ya awali kwa kujumuisha hash ya risiti ya awali katika mzigo mpya wa risiti.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ikiwa mtu yeyote ataondoa au kupanga upya risiti, mnyororo huvunjika mahali hapo kabisa. Uthibitishaji wa risiti yoyote baadaye hutofa kwa sababu `previous_receipt_hash` haendani tena na hash halisi ya ile iliyotangulia.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Sasa vunja mnyororo kwa kubadilisha risiti ya katikati na uthibitishe tena. Risiti iliyobadilishwa inashindwa ukaguzi wake wa saini, NA risiti inayofuata inashindwa ukaguzi wa kiungo cha mnyororo wake (kwa sababu `previous_receipt_hash` haisi tena kulingana na hash ya risiti ya katikati iliyobadilishwa).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Risiti 0 bado inathibitishwa (haikubadilishwa na haina risiti ya awali ya kutegemea). Risiti 1 inashindwa ukaguzi wake wa saini kwa sababu tulibadilisha `tool_args_hash`. Risiti 2 inashindwa ukaguzi wa kiungo cha mnyororo kwa sababu `previous_receipt_hash` yake ilihesabiwa dhidi ya risiti ya awali (sasa iliyobadilishwa) 1.

Hata kama mshambuliaji ata-saini tena risiti iliyobadilishwa 1 (ambayo hawawezi kufanya bila ufunguo binafsi), kutokufanana kwa kiungo cha mnyororo katika risiti 2 bado kutafichua kuingilia kati. Ili kuficha mabadiliko, mshambuliaji angehitaji kusaini tena kila risiti kuanzia mahali pa mabadiliko, jambo ambalo linahitaji kuwa na ufunguo binafsi.


## Sehemu ya 4: Vikosoa kwa simu ya chombo cha wakala kwa kusaini risiti

Katika usambazaji halisi, hutaki kila mwandishi wa wakala kukumbuka kupiga simu `make_receipt`. Unataka kusaini risiti kuwa moja kwa moja kwa kila unyonyaji wa chombo.

Hapa kuna mfano rahisi kabisa: darasa la kufunika linalochukua kazi yoyote ya chombo inayoweza kupigwa na kurudisha toleo la kuemitia risiti. Hii inaendana na mfumo wowote wa wakala, ikiwa ni pamoja na Microsoft Agent Framework (`agent_framework.foundry`).

Ikiwa huna mradi wa Microsoft Foundry uliowekwa, bandia ya ndani hapo chini bado inaonyesha mfano huu.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Kuingiza na Mfumo wa Wakala wa Microsoft  

Kifunguzo cha `ReceiptedTool` kilicho hapo juu hakihusiani na mfumo wowote maalumu. Ili kukitumia ndani ya wakala aliyejengwa kwa Mfumo wa Wakala wa Microsoft, jisajili kazi iliyofunikwa kama chombo. Rasimu (utabadilisha mfano wa uongo na usajili halisi wa chombo cha Microsoft Foundry):  

```python
# Pseudocode inaonyesha muundo wa muunganiko.
# import os
# kutoka agent_framework.foundry ingiza FoundryChatClient
# kutoka azure.identity ingiza AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     maelekezo="Wewe ni wakala wa Contoso Travel ...",
#     tools=[receipted_lookup],   # chombo kilichofunikwa, sio kazi ghafi
# )
# response = agent.run("Tafuta ndege kutoka Sydney hadi Los Angeles mwezi Juni.")
#
# # Baada ya utekelezaji, kila simu ya chombo iliyo fanywa na wakala ina risiti iliyo saini:
# audit_chain = receipted_lookup.receipts
```
  
Mfumo wa wakala hauitaji kujua chochote kuhusu risiti. Kusaini risiti kunafunikwa karibu na chombo, sio kuunganishwa kamepesi ndani ya mfumo. Hivi ndivyo unavyoongeza asili kwenye msimbo ulio tayari wa wakala bila kuandika upya wakala.  


## Muhtasari na changamoto ya kupanua

Umefanya:

- Kuunda jozi ya funguo za Ed25519.
- Kujenga na kusaini risiti kwa ajili ya wito la chombo cha mawakala.
- Kuhakiki risiti bila mtandao kwa kutumia funguo ya umma pekee.
- Kubadilisha risiti na kuona uthibitishaji ukishindwa.
- Kujenga mfululizo wa risiti tatu zenye mnyororo wa hash.
- Kubadilisha sehemu ya katikati ya mnyororo na kuona kushindwa kwa saini na kushindwa kwa kiungo cha mnyororo.
- Kufunga kazi ya chombo na kusaini risiti moja kwa moja.

**Changamoto ya kupanua.** Panua muundo wa risiti kwa kuingiza kipengele cha `request_id` (UUID kwa ufuatiliaji wa kusambazwa). Sasisha `make_receipt` ili kiingizwe, na thibitisha risiti bado zinathibitishwa kuanzia mwisho hadi mwisho. Kisha badilisha kipengele baada ya kusaini na thibitisha uthibitishaji unashindwa. Hii itakulazimisha kuelewa jinsi kila biti ya usimbaji wa kanoni inavyosaidia saini.

**Mkupuo muhimu.** Risiti zinaonyesha vitu vitatu tu, na vitu vitatu tu: utambulisho (funguo hii ilisaini maudhui haya), uadilifu (maudhui hayajabadilika tangu saini), na upangaji (risiti hii ilifuata risiti ile ile). Haziathibitishi kuwa kitendo cha mwakala kilikuwa sahihi, kwamba sera iliyoitwa `policy_id` ilipimwa kweli, au kwamba mwakala alifuata kila sheria. Risiti ni msingi. Utawala ni mfumo unaojengwa juu yake.

Soma README ya somo tena ukiwa na mkupuo huo akilini. Makosa yanayojirudia mara nyingi kwa timu kuhusu risiti ni kudhani "tunayo risiti" inamaanisha "tunatawaliwa." Hilo si kweli. Risiti zinafanya tabia ya mwakala isiwe ya siri kwa ukaguzi. Hazifanya iwe sahihi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
